# TCSPC acquisition loader and behavior-camera viewer

This notebook has three steps: load and validate a paired `.spc`/`.set` acquisition, build one analysis-ready event DataFrame, and inspect photon counts and TCSPC decays by behavior-camera frame.

## Why this workflow

Fiber-photometry systems commonly align neural signals to behavior using hardware TTL edges and then analyze signals on the behavioral timebase. Here the camera Marker-2 edges define that timebase, while TCSPC preserves two measurements that analog photometry does not: photon counts and photon microtimes. Raw marker edges are always retained; any edge filtering used to identify camera frames is an explicit, reviewable analysis choice.

References: [pyPhotometry synchronization](https://pyphotometry.readthedocs.io/en/stable/user-guide/importing-data/#synchronisation) and [Akam et al., *Lights, fiber, action!*](https://doi.org/10.1016/j.neuron.2023.11.012).

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import shutil
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from spc150_fifo_parser import (
    assign_photons_to_frames,
    build_unified_event_dataframe,
    infer_marker_debounce_ticks,
    parse_spc150_fifo,
    parse_spc_header,
    require_set_metadata,
    resolve_sync_rate_hz,
    validate_acquisition_pair,
)

## 1. Point the loader at one acquisition

`FRAME_EDGE_FILTER = "auto"` reproduces the camera-edge selection used for this test recording. Set it to `None` to treat every Marker-2 edge as a camera frame, or to an integer number of macrotime ticks to use a fixed, acquisition-validated debounce threshold.

In [ ]:
SPC_PATH = Path("test_camera_30fps.spc")
SET_PATH = Path("test_camera_30fps.set")
VIDEO_PATH = None  # e.g. Path("behavior_camera.mp4")

CAMERA_MARKER_BIT = 2
FRAME_EDGE_FILTER = "auto"  # "auto", None, or a non-negative tick threshold
FAIL_ON_GAP = True

FRAME_TO_VIEW = 2
OVERVIEW_ROLLING_FRAMES = 30  # ~1 s smoothing at 30 fps; use 1 for raw counts

In [ ]:
@dataclass
class TCSPCAcquisition:
    spc_path: Path
    set_path: Path
    metadata: object
    header: object
    arrays: dict
    marker_events: list
    events: pd.DataFrame
    sync_rate_hz: float
    qc: pd.DataFrame


def load_tcspc_acquisition(spc_path, set_path, *, fail_on_gap=True):
    """Load a paired acquisition and fail on format, pairing, or timing corruption."""
    spc_path = Path(spc_path).expanduser()
    set_path = Path(set_path).expanduser()
    if spc_path.suffix.lower() != ".spc":
        raise ValueError(f"Expected an .spc file: {spc_path}")
    if set_path.suffix.lower() != ".set":
        raise ValueError(f"Expected a .set file: {set_path}")

    metadata = require_set_metadata(set_path)
    validate_acquisition_pair(spc_path, metadata)
    arrays, marker_events, header_word = parse_spc150_fifo(spc_path)
    header = parse_spc_header(header_word)
    sync_rate_hz, sync_source = resolve_sync_rate_hz(
        None, metadata.spc_params, header_word
    )
    if sync_rate_hz is None:
        raise ValueError("No valid macrotime clock was found in the SPC header.")

    events = build_unified_event_dataframe(arrays, sync_rate_hz)
    photon_rows = events["type"].eq("photon")
    tac_bin_width_s = metadata.spc_params.get("SP_TAC_TC")
    has_microtime_calibration = (
        isinstance(tac_bin_width_s, (int, float)) and tac_bin_width_s > 0
    )
    events["microtime_s"] = np.nan
    if has_microtime_calibration:
        events.loc[photon_rows, "microtime_s"] = (
            events.loc[photon_rows, "nanotime_bin"].astype(float) * float(tac_bin_width_s)
        )
    events["event_time_s"] = events["macrotime_s"]
    events.loc[photon_rows, "event_time_s"] += events.loc[photon_rows, "microtime_s"].fillna(0)

    valid_ticks = arrays["macrotime_ticks"][arrays["macrotime_valid"]]
    backward_steps = int(np.sum(np.diff(valid_ticks) < 0))
    gap_records = int(np.sum(arrays["has_gap"]))
    out_of_range_nanotimes = int(
        np.sum((arrays["nanotime_bin"][arrays["is_photon"]] < 0)
               | (arrays["nanotime_bin"][arrays["is_photon"]] > 4095))
    )
    out_of_range_channels = int(
        np.sum(arrays["detector_channel"][arrays["is_photon"]] >= header.routing_channels)
    )

    critical_errors = []
    if backward_steps:
        critical_errors.append(f"{backward_steps} backward macrotime step(s)")
    if out_of_range_nanotimes:
        critical_errors.append(f"{out_of_range_nanotimes} invalid photon microtime(s)")
    if out_of_range_channels:
        critical_errors.append(f"{out_of_range_channels} invalid detector channel(s)")
    if fail_on_gap and gap_records:
        critical_errors.append(f"{gap_records} FIFO GAP/data-loss record(s)")
    if critical_errors:
        raise ValueError("Acquisition integrity checks failed: " + "; ".join(critical_errors))

    qc = pd.DataFrame(
        [
            ("SPC/SET filenames match", True, spc_path.name),
            ("Module", True, metadata.module),
            ("Macrotime clock", sync_rate_hz > 0, f"{sync_rate_hz:g} Hz ({sync_source})"),
            ("Valid photon records", True, int(np.sum(arrays["is_photon"]))),
            ("Invalid photon records", True, int(np.sum(arrays["is_invalid_photon"]))),
            ("Marker edge records", len(marker_events) > 0, len(marker_events)),
            ("Overflow wraps reconstructed", True, int(np.sum(arrays["overflow_wraps"]))),
            ("Backward valid timestamps", backward_steps == 0, backward_steps),
            ("FIFO GAP/data-loss records", gap_records == 0, gap_records),
            ("Photon microtime calibration", has_microtime_calibration,
             f"{float(tac_bin_width_s):.6g} s/bin" if has_microtime_calibration else "missing"),
        ],
        columns=["check", "passed", "value"],
    )
    return TCSPCAcquisition(
        spc_path, set_path, metadata, header, arrays, marker_events,
        events, float(sync_rate_hz), qc
    )

In [ ]:
acquisition = load_tcspc_acquisition(
    SPC_PATH,
    SET_PATH,
    fail_on_gap=FAIL_ON_GAP,
)
display(acquisition.qc)

if not acquisition.qc["passed"].all():
    print("Review the failed non-critical checks before analysis.")

## 2. Analysis-ready event DataFrame

`events_df` contains every valid photon and every raw marker edge. `macrotime_s` is shared by photons and markers. Photons additionally have calibrated `microtime_s`; `event_time_s` adds that within-sync-period delay to the photon macrotime. Marker-specific and photon-specific fields remain null where they do not apply.

In [ ]:
events_df = acquisition.events.copy()

display(events_df.head(10))
display(events_df.groupby("type", observed=True).size().rename("records").to_frame())

timestamp_columns = [
    "record_index", "type", "event_time_s", "macrotime_s", "microtime_s",
    "detector_channel_1based", "marker_mask", "marker0", "marker1",
    "marker2", "marker3", "has_overflow", "has_gap",
]
display(events_df[timestamp_columns].head(20))

## Build the behavior-camera frame table

This step converts the chosen marker edges into camera-frame windows. The raw marker events in `events_df` are never modified. Automatic filtering is provided only for this acquisition pattern and reports the chosen threshold.

In [ ]:
def build_camera_frame_table(acquisition, *, marker_bit=2, edge_filter=None):
    marker_column = f"marker{marker_bit}"
    marker_ticks = acquisition.events.loc[
        acquisition.events[marker_column].fillna(False), "macrotime_ticks"
    ].to_numpy(dtype=np.int64)
    if marker_ticks.size < 2:
        raise ValueError(f"Need at least two Marker-{marker_bit} edges to define frames.")

    if edge_filter == "auto":
        debounce_ticks = infer_marker_debounce_ticks(marker_ticks)
    elif edge_filter is None:
        debounce_ticks = None
    elif isinstance(edge_filter, (int, np.integer)) and edge_filter >= 0:
        debounce_ticks = int(edge_filter)
    else:
        raise ValueError('edge_filter must be "auto", None, or a non-negative integer.')

    framed_events, frame_table = assign_photons_to_frames(
        acquisition.events,
        marker_id=marker_bit,
        marker_debounce_ticks=debounce_ticks,
    )
    framed_events["camera_frame"] = framed_events["frame_index"]
    frame_table["camera_frame"] = frame_table["frame_index"]
    frame_table["frame_duration_s"] = frame_table["frame_end_s"] - frame_table["frame_start_s"]

    complete = frame_table["frame_duration_s"].dropna()
    frame_rate_hz = 1.0 / complete.mean() if not complete.empty else np.nan
    diagnostics = pd.Series(
        {
            "Raw camera-marker edges": marker_ticks.size,
            "Selected camera frames": len(frame_table),
            "Debounce threshold (ticks)": debounce_ticks,
            "Measured camera rate (Hz)": frame_rate_hz,
            "First frame time (s)": frame_table["frame_start_s"].iloc[0],
            "Last frame time (s)": frame_table["frame_start_s"].iloc[-1],
        },
        name="value",
    ).to_frame()
    return framed_events, frame_table, diagnostics

In [ ]:
framed_events_df, frame_table_df, frame_diagnostics = build_camera_frame_table(
    acquisition,
    marker_bit=CAMERA_MARKER_BIT,
    edge_filter=FRAME_EDGE_FILTER,
)
display(frame_diagnostics)
display(frame_table_df.head())

## Validate the saved SpinView MP4 with FFprobe

FFprobe counts decoded frames in display order. The notebook maps the first decoded MP4 frame to the first selected TCSPC camera trigger and uses zero-based indexing internally. MP4 presentation timestamps are reported for diagnostics only; the TCSPC trigger timestamps remain authoritative.

Set `VIDEO_PATH` above to enable this check. The workflow stops if FFprobe is unavailable, the MP4 is unreadable, or its decoded-frame count differs from the selected trigger count.

In [ ]:
def ffprobe_video(video_path, *, ffprobe_executable="ffprobe"):
    """Return decoded-frame and stream metadata from the first MP4 video stream."""
    video_path = Path(video_path).expanduser()
    if not video_path.is_file():
        raise FileNotFoundError(f"Video file not found: {video_path}")
    executable = shutil.which(ffprobe_executable)
    if executable is None:
        raise RuntimeError(
            "ffprobe was not found on PATH. Install FFmpeg, restart the kernel, "
            "and confirm that `ffprobe -version` works in a terminal."
        )

    command = [
        executable,
        "-v", "error",
        "-select_streams", "v:0",
        "-count_frames",
        "-show_entries",
        "stream=index,codec_name,width,height,avg_frame_rate,r_frame_rate,duration,nb_frames,nb_read_frames",
        "-of", "json",
        str(video_path),
    ]
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        detail = result.stderr.strip() or "unknown FFprobe error"
        raise ValueError(f"FFprobe could not read {video_path}: {detail}")
    try:
        payload = json.loads(result.stdout)
    except json.JSONDecodeError as exc:
        raise ValueError("FFprobe returned invalid JSON.") from exc

    streams = payload.get("streams", [])
    if len(streams) != 1:
        raise ValueError(f"Expected one selected video stream, found {len(streams)}.")
    stream = streams[0]
    frame_text = stream.get("nb_read_frames")
    try:
        decoded_frames = int(frame_text)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"FFprobe did not report a decoded frame count: {frame_text!r}") from exc
    if decoded_frames <= 0:
        raise ValueError(f"FFprobe reported an invalid frame count: {decoded_frames}")

    def parse_rate(value):
        try:
            numerator, denominator = str(value).split("/", 1)
            return float(numerator) / float(denominator) if float(denominator) else np.nan
        except (TypeError, ValueError):
            return np.nan

    return {
        "video_path": str(video_path),
        "decoded_frames": decoded_frames,
        "codec": stream.get("codec_name"),
        "width": stream.get("width"),
        "height": stream.get("height"),
        "nominal_frame_rate_hz": parse_rate(stream.get("avg_frame_rate")),
        "reported_frame_rate_hz": parse_rate(stream.get("r_frame_rate")),
        "duration_s": pd.to_numeric(stream.get("duration"), errors="coerce"),
        "container_nb_frames": pd.to_numeric(stream.get("nb_frames"), errors="coerce"),
    }


def validate_video_trigger_alignment(video_path, frame_table):
    """Require a one-to-one mapping between decoded MP4 frames and TCSPC triggers."""
    info = ffprobe_video(video_path)
    tcspc_frames = int(len(frame_table))
    delta = int(info["decoded_frames"] - tcspc_frames)
    diagnostics = pd.Series(
        {
            "Video": info["video_path"],
            "Codec": info["codec"],
            "Dimensions": f'{info["width"]} × {info["height"]}',
            "Decoded MP4 frames": info["decoded_frames"],
            "Selected TCSPC triggers": tcspc_frames,
            "Frame-count difference": delta,
            "MP4 nominal rate (Hz)": info["nominal_frame_rate_hz"],
            "TCSPC measured rate (Hz)": 1.0 / frame_table["frame_duration_s"].dropna().mean(),
        },
        name="value",
    ).to_frame()
    if delta != 0:
        raise ValueError(
            f"Video/TCSPC frame mismatch: MP4 has {info['decoded_frames']} decoded frames "
            f"but TCSPC has {tcspc_frames} selected triggers (difference {delta:+d}). "
            "Check camera arming, recording start/stop, dropped frames, and marker filtering."
        )

    validated = frame_table.copy()
    validated["video_frame_0based"] = np.arange(tcspc_frames, dtype=np.int64)
    validated["video_frame_1based"] = validated["video_frame_0based"] + 1
    return validated, diagnostics, info

In [ ]:
if VIDEO_PATH is not None:
    frame_table_df, video_diagnostics, video_info = validate_video_trigger_alignment(
        VIDEO_PATH, frame_table_df
    )
    display(video_diagnostics)
else:
    video_info = None
    frame_table_df["video_frame_0based"] = frame_table_df["camera_frame"].astype("int64")
    frame_table_df["video_frame_1based"] = frame_table_df["video_frame_0based"] + 1
    print("VIDEO_PATH is None: FFprobe validation skipped.")

## 3. Data viewer

The first panel shows raw photon counts per camera frame for every detector channel. The second shows the selected frame's channel counts. The third shows the selected frame's TCSPC microtime distribution; a single 30 fps frame may contain too few photons for a reliable lifetime fit, so fitting should later aggregate a defined behavioral epoch or validated frame window.

In [ ]:
def plot_tcspc_viewer(
    framed_events,
    frame_table,
    frame_number,
    *,
    rolling_frames=1,
    decay_bins=64,
):
    if frame_number not in set(frame_table["camera_frame"].astype(int)):
        raise ValueError(
            f"camera frame {frame_number} is unavailable; choose "
            f"{int(frame_table.camera_frame.min())}..{int(frame_table.camera_frame.max())}"
        )
    if rolling_frames < 1:
        raise ValueError("rolling_frames must be >= 1")

    photons = framed_events.loc[
        (framed_events["type"] == "photon") & framed_events["camera_frame"].notna()
    ].copy()
    photons["camera_frame"] = photons["camera_frame"].astype(int)
    photons["channel"] = photons["detector_channel_1based"].astype(int)

    all_frames = frame_table["camera_frame"].astype(int).to_numpy()
    channels = np.sort(photons["channel"].unique())
    counts = (
        photons.groupby(["camera_frame", "channel"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(index=all_frames, columns=channels, fill_value=0)
    )
    plotted_counts = (
        counts.rolling(rolling_frames, center=True, min_periods=1).mean()
        if rolling_frames > 1 else counts
    )
    frame_times = frame_table.set_index("camera_frame").loc[all_frames, "frame_start_s"]

    selected = photons.loc[photons["camera_frame"] == int(frame_number)]
    selected_counts = selected.groupby("channel", observed=True).size().reindex(channels, fill_value=0)
    selected_time = float(
        frame_table.loc[frame_table["camera_frame"] == frame_number, "frame_start_s"].iloc[0]
    )

    colors = plt.get_cmap("tab20")(np.linspace(0, 1, max(len(channels), 1)))
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), constrained_layout=True)

    for color, channel in zip(colors, channels):
        axes[0].plot(
            frame_times.to_numpy(), plotted_counts[channel].to_numpy(),
            color=color, linewidth=0.9, alpha=0.85, label=f"Ch {channel}",
        )
    axes[0].axvline(selected_time, color="black", linewidth=1, alpha=0.7)
    axes[0].set(
        title="All detector channels on the behavior-camera timebase",
        xlabel="Acquisition time (s)",
        ylabel=("Photons/frame" if rolling_frames == 1 else f"Photons/frame ({rolling_frames}-frame mean)"),
    )
    axes[0].legend(ncol=4, fontsize=8, frameon=False)

    axes[1].bar(channels, selected_counts.to_numpy(), color=colors)
    axes[1].set(
        title=f"Camera frame {frame_number}: {len(selected)} photons by detector channel",
        xlabel="Detector channel (1-based)",
        ylabel="Photon count",
        xticks=channels,
    )

    has_calibrated_microtime = selected["microtime_s"].notna().any()
    for color, channel in zip(colors, channels):
        channel_photons = selected.loc[selected["channel"] == channel]
        if channel_photons.empty:
            continue
        values = (
            channel_photons["microtime_s"].to_numpy(dtype=float) * 1e9
            if has_calibrated_microtime
            else channel_photons["nanotime_bin"].to_numpy(dtype=float)
        )
        axes[2].hist(
            values, bins=decay_bins, histtype="step", linewidth=1.4,
            color=color, label=f"Ch {channel}",
        )
    axes[2].set(
        title=f"Camera frame {frame_number}: TCSPC microtime distribution",
        xlabel=("Microtime (ns)" if has_calibrated_microtime else "Microtime bin"),
        ylabel="Photons/bin",
    )
    if not selected.empty:
        axes[2].legend(ncol=4, fontsize=8, frameon=False)
    for axis in axes:
        axis.grid(alpha=0.2)
    return fig, axes

In [ ]:
fig, axes = plot_tcspc_viewer(
    framed_events_df,
    frame_table_df,
    FRAME_TO_VIEW,
    rolling_frames=OVERVIEW_ROLLING_FRAMES,
)
plt.show()

## Joining scored behavior later

Use `camera_frame` as the stable join key, rather than matching independent computer timestamps. First verify that saved-video frame count and trigger count agree; dropped frames or a camera start offset must be resolved before a one-to-one merge. A behavior table with one row per validated video frame can then be merged directly:

```python
behavior_df = pd.read_csv("behavior_by_frame.csv")  # must contain camera_frame
behavior_tcspc_df = frame_table_df.merge(
    behavior_df,
    on="camera_frame",
    how="left",
    validate="one_to_one",
)
```

Typical photometry preprocessing—bleaching correction, control-channel regression, normalization, and peri-event averaging—should be added only after defining which detector or excitation channels are signal and controls. For TCSPC, lifetime estimation should use photon microtime histograms aggregated over predeclared behavioral windows and should include the instrument response function and background model.